In [2]:
import pandas as pd
import numpy as np

In [3]:
data = pd.read_csv('data/GSE115469_Data.csv', index_col=0)
cell_cluster_data = pd.read_csv('data/GSE115469_CellClusterType.txt', sep='\t')

# Pretprocesiranje - opsti koraci

In [4]:
gene_frequency = (data > 0).mean(axis=1)
data = data.loc[gene_frequency >= 0.05]

In [ ]:
gene_var = data.var(axis=1)
gene_var_sorted = gene_var.sort_values(ascending=False)
gene_var_keep = gene_var_sorted[:2000]

In [6]:
data = data.loc[gene_var_keep.index]

In [7]:
data = data.clip(upper=data.quantile(0.999).max())

In [8]:
data = data.T

# PRETPROCESIRANJE - Klasifikacija

## Korak 1: Spajanje tabela

In [9]:
data.index.name = 'CellName'
data.reset_index(inplace=True)

In [10]:
data.head()

,CellName,APOC3,APOA2,ORM1,HP,ALB,APOC1,TTR,SAA1,APOA1,...,HMGCR,RNF149,GIMAP1,APH1A,MAF1,TACC1,PDXDC1,FUNDC2,RNF167,PSAT1
0,P1TLH_AAACCTGAGCAGCCTC_1,0.836165,0.836165,0.000000,1.362102,0.000000,0.836165,0.000000,1.362102,0.836165,...,0.0,0.000000,0.000000,0.000000,0.0,0.836165,0.0,0.000000,0.0,0.0
1,P1TLH_AAACCTGTCCTCATTA_1,1.436498,1.436498,3.458904,1.149924,4.797971,2.059859,0.572995,2.493720,1.300315,...,0.0,0.000000,0.000000,0.314760,0.0,0.982011,0.0,0.000000,0.0,0.0
2,P1TLH_AAACCTGTCTAAGCCA_1,1.820614,0.000000,0.000000,1.180248,0.000000,0.000000,1.180248,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,1.180248,0.0,1.180248,0.0,0.0
3,P1TLH_AAACGGGAGTAGGCCA_1,0.000000,0.644905,0.000000,0.000000,0.000000,0.000000,0.000000,1.428094,0.000000,...,0.0,1.428094,0.644905,0.644905,0.0,0.000000,0.0,0.000000,0.0,0.0
4,P1TLH_AAACGGGGTTCGGGCT_1,1.284221,1.284221,0.780522,1.656843,1.656843,0.780522,0.000000,1.656843,0.780522,...,0.0,0.000000,1.284221,0.780522,0.0,0.000000,0.0,0.000000,0.0,0.0


In [11]:
merged_data = pd.merge(
    data,
    cell_cluster_data,
    on='CellName'
)

In [12]:
merged_data.columns

Index(['CellName', 'APOC3', 'APOA2', 'ORM1', 'HP', 'ALB', 'APOC1', 'TTR',
       'SAA1', 'APOA1',
       ...
       'MAF1', 'TACC1', 'PDXDC1', 'FUNDC2', 'RNF167', 'PSAT1', 'Sample',
       'Cell#', 'Cluster#', 'CellType'],
      dtype='object', length=2005)

## Korak 2: Izdvajanje feature-a i target-a

In [13]:
X = merged_data.drop(columns=['CellName', 'Cell#', 'CellType', 'Cluster#', 'Sample'])
y = merged_data['CellType']

## Korak 3: Kodiranje target promenljive

In [14]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

## Korak 4: Skaliranje

In [15]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

## Korak 4: Feature Selection

Postoji mnogo više gena nego ćelija, biramo podskup najinformativnijih gena.

In [16]:
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(
    score_func=f_classif,
    k=500
)

X_selected = selector.fit_transform(X_scaled, y)

## Korak 6: Train/test split

In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_selected,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# SVM

In [ ]:
from sklearn.svm import SVC

svm = SVC(
    kernel="rbf",
    random_state=42
)

svm.fit(X_train, y_train)

pred_svm = svm.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, pred_svm))

              precision    recall  f1-score   support

           0       0.97      0.98      0.98        61
           1       1.00      1.00      1.00        24
           2       0.93      0.74      0.82        19
           3       1.00      0.86      0.92         7
           4       0.91      0.90      0.90       201
           5       0.92      0.94      0.93       182
           6       0.85      1.00      0.92       126
           7       0.91      0.92      0.91       121
           8       1.00      0.90      0.95        40
           9       0.92      0.80      0.86        30
          10       0.97      0.98      0.98       163
          11       1.00      1.00      1.00        26
          12       0.98      0.96      0.97        98
          13       1.00      0.95      0.97        76
          14       0.98      0.95      0.97        65
          15       0.99      0.98      0.99       102
          16       0.98      1.00      0.99        42
          17       0.97    